In [10]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ankitdata1/moesi-v2/mosei_aligned_seq50_v2.pt
/kaggle/input/datasets/ankitdata1/mosei-aligned/mosei_aligned_seq50.pt
/kaggle/input/datasets/ankitdata1/mosei-aligned/metadata.csv


In [11]:
# ============================================================
# CELL 1 — Imports, Config, Dataset, Encoders
# ============================================================

import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# ── Config ──────────────────────────────────────────────────

# In Config, update these:
class Config:
    data_path   = "/kaggle/input/datasets/ankitdata1/moesi-v2/mosei_aligned_seq50_v2.pt"
    device       = "cuda" if torch.cuda.is_available() else "cpu"
    hidden_dim   = 128   
    latent_dim   = 128
    num_classes  = 3
    batch_size   = 128
    epochs       = 100
    lr           = 5e-4
    weight_decay = 3e-4
    grad_clip    = 1.0
    seed         = 42
    lam_diff     = 0.0   # even quieter — classifier must dominate
    lam_recon    = 0.05
    lam_causal   = 0.01
    lam_kl_beta  = 0.3
    warmup_aux   = 50     # auxiliary losses fully ramp at epoch 50
    warmup_kl    = 60
    free_bits    = 0.25
    noise_std    = 0.02
    mask_prob    = 0.15
    drop_prob    = 0.20
    T            = 1000
    ddim_steps   = 50
    cfg_scale    = 4.0
    cfg_dropout  = 0.20

cfg = Config()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

# ── Dataset ─────────────────────────────────────────────────
# Data is (N, 50, D) — full temporal sequences

class MOSEIDataset(Dataset):
    def __init__(self, text, audio, video, labels, augment=False):
        self.text   = torch.tensor(text,   dtype=torch.float32)
        self.audio  = torch.tensor(audio,  dtype=torch.float32)
        self.video  = torch.tensor(video,  dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text  = torch.nan_to_num(self.text[idx].clone())
        audio = torch.nan_to_num(self.audio[idx].clone())
        video = torch.nan_to_num(self.video[idx].clone())
        label = self.labels[idx]

        if self.augment:
            audio = audio + torch.randn_like(audio) * cfg.noise_std
            video = video + torch.randn_like(video) * cfg.noise_std
            text  = text  * (torch.rand(text.shape[0],  1) > cfg.mask_prob).float()
            audio = audio * (torch.rand(audio.shape[0], 1) > cfg.mask_prob).float()
            video = video * (torch.rand(video.shape[0], 1) > cfg.mask_prob).float()
            if torch.rand(1).item() < cfg.drop_prob:
                text  = torch.zeros_like(text)
            if torch.rand(1).item() < cfg.drop_prob:
                audio = torch.zeros_like(audio)
            if torch.rand(1).item() < cfg.drop_prob:
                video = torch.zeros_like(video)

        return text, audio, video, label


# ── Updated get_loaders ──────────────────────────────────────
# Updated remap — simply drop Other samples entirely
def remap_labels(labels):
    # Keep only Happy(0), Sad(1), Angry(2)
    # Return remapped labels AND a mask for valid samples
    mask = labels <= 2
    remapped = labels.copy()
    classes = ["Happy", "Sad", "Angry"]
    counts = np.bincount(remapped[mask], minlength=3)
    for c, n in zip(classes, counts):
        print(f"  {c}: {n} ({100*n/mask.sum():.1f}%)")
    return remapped, mask

from torch.utils.data import WeightedRandomSampler
# Updated get_loaders to apply mask
def get_loaders(data_path):
    d = torch.load(data_path, map_location="cpu", weights_only=False)

    text   = np.array(d["text"])
    audio  = np.array(d["audio"])
    video  = np.array(d["vision"])
    labels, mask = remap_labels(np.array(d["labels"]))

    # Apply mask before splitting
    text, audio, video, labels = text[mask], audio[mask], video[mask], labels[mask]

    N = len(labels)
    rng = np.random.default_rng(cfg.seed)
    idx = rng.permutation(N)

    n_train = int(0.70 * N)
    n_val   = int(0.15 * N)
    i_train = idx[:n_train]
    i_val   = idx[n_train:n_train + n_val]
    i_test  = idx[n_train + n_val:]

    train_ds = MOSEIDataset(text[i_train], audio[i_train], video[i_train], labels[i_train], augment=True)
    val_ds   = MOSEIDataset(text[i_val],   audio[i_val],   video[i_val],   labels[i_val],   augment=False)
    test_ds  = MOSEIDataset(text[i_test],  audio[i_test],  video[i_test],  labels[i_test],  augment=False)

    # Sqrt oversampling for Sad and Angry
    train_labels = labels[i_train]
    counts = np.bincount(train_labels, minlength=cfg.num_classes).astype(float)
    class_prob = 1.0 / (np.sqrt(counts) + 1e-6)
    sample_weights = torch.tensor(class_prob[train_labels], dtype=torch.float32)
    sampler = WeightedRandomSampler(sample_weights, len(train_ds), replacement=True)

    kw = dict(batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
    return (
        DataLoader(train_ds, sampler=sampler, **kw),
        DataLoader(val_ds,   shuffle=False,   **kw),
        DataLoader(test_ds,  shuffle=False,   **kw),
    )

# ── Updated ModalityEncoder ───────────────────────────────────
# Input is now (B, 50, D) — mean-pool over time then project.
class ModalityEncoder(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        H = cfg.hidden_dim
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, H),
            nn.LayerNorm(H),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=H,
            nhead=4,
            dim_feedforward=H,        # was H*2, reduce capacity
            dropout=0.2,              # was 0.1
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, H) * 0.02)

    def forward(self, x):
        B = x.shape[0]
        h = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        h = torch.cat([cls, h], dim=1)
        h = self.transformer(h)
        return h[:, 0, :]

In [12]:
# ============================================================
# Causal Attention Graph + VAE Bottleneck
# ============================================================

# ── Causal Attention Graph ───────────────────────────────────
# Learns a soft directed adjacency matrix A over the 3 modality
# nodes {T, A, V}. Column sums give per-modality importance
# weights w fed to the U-Net. NOTEARS penalty enforces acyclicity.

class CausalAttentionGraph(nn.Module):
    def __init__(self):
        super().__init__()
        H = cfg.hidden_dim
        self.W_Q = nn.Linear(H, H, bias=False)
        self.W_K = nn.Linear(H, H, bias=False)
        self.scale = H ** 0.5
        # mask to zero self-loops — shape (3, 3)
        self.register_buffer(
            "self_loop_mask",
            1 - torch.eye(3)
        )

    def forward(self, h_t, h_a, h_v):
        # h_t, h_a, h_v : (B, H)
        # Stack into node matrix N : (B, 3, H)
        N = torch.stack([h_t, h_a, h_v], dim=1)

        Q = self.W_Q(N)   # (B, 3, H)
        K = self.W_K(N)   # (B, 3, H)

        # Directed edge scores : (B, 3, 3)
        # S[i,j] = influence of node j on node i
        S = torch.bmm(Q, K.transpose(1, 2)) / self.scale

        # Mask self-loops then sigmoid → soft adjacency A ∈ [0,1]
        A = torch.sigmoid(S) * self.self_loop_mask.unsqueeze(0)

        # Per-modality importance weights : column sums → (B, 3)
        w = A.sum(dim=1)   # how much each modality is pointed-to

        # NOTEARS acyclicity penalty : h(A) = tr(e^{A◦A}) - d
        # Averaged over batch
        A2 = A * A                          # Hadamard square (B,3,3)
        # Matrix exponential approximation via first-order Taylor is
        # unstable; use the exact penalty on mean A for efficiency
        A_mean = A.mean(dim=0)              # (3, 3)
        expm   = torch.linalg.matrix_exp(A_mean * A_mean)
        L_causal = (expm.trace() - 3).abs()

        return A, w, L_causal


# ── VAE Bottleneck ───────────────────────────────────────────
# Takes the 3 weighted modality vectors, concatenates them,
# and maps to (mu, log_var) in latent_dim space.
# Free-bits KL prevents posterior collapse.

class VAEBottleneck(nn.Module):
    def __init__(self):
        super().__init__()
        H  = cfg.hidden_dim
        dz = cfg.latent_dim
        # 3 modality vectors weighted and concatenated → 3H
        self.fusion = nn.Sequential(
            nn.Linear(3 * H, H),
            nn.LayerNorm(H),
            nn.GELU(),
        )
        self.mu_head      = nn.Linear(H, dz)
        self.logvar_head  = nn.Linear(H, dz)

    def encode(self, h_t, h_a, h_v, w):
        # w : (B, 3) importance weights from causal graph
        # Weight each modality vector before fusion
        h_t = h_t * w[:, 0:1]
        h_a = h_a * w[:, 1:2]
        h_v = h_v * w[:, 2:3]

        x   = torch.cat([h_t, h_a, h_v], dim=-1)  # (B, 3H)
        h   = self.fusion(x)                        # (B, H)

        mu      = self.mu_head(h)
        log_var = self.logvar_head(h).clamp(-10, 5)
        return mu, log_var

    def reparameterize(self, mu, log_var):
        if self.training:
            std = (0.5 * log_var).exp()
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu   # deterministic at inference

    def kl_loss(self, mu, log_var):
        # Per-dimension KL
        kl_per_dim = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())
        # Free-bits: only penalise dimensions above threshold
        kl_per_dim = torch.clamp(kl_per_dim, min=cfg.free_bits)
        return cfg.lam_kl_beta * kl_per_dim.mean()

    def forward(self, h_t, h_a, h_v, w):
        mu, log_var = self.encode(h_t, h_a, h_v, w)
        z           = self.reparameterize(mu, log_var)
        L_kl        = self.kl_loss(mu, log_var)
        return z, mu, log_var, L_kl


# ── Reconstruction Decoder ───────────────────────────────────
# Auxiliary path only. Decodes z back to each modality's
# original feature dim to keep the latent space informative.

class ReconDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        dz = cfg.latent_dim

        def _dec(out_dim):
            return nn.Sequential(
                nn.Linear(dz, cfg.hidden_dim),
                nn.LayerNorm(cfg.hidden_dim),
                nn.GELU(),
                nn.Dropout(0.1),
                nn.Linear(cfg.hidden_dim, out_dim),
            )

        self.dec_t = _dec(300)
        self.dec_a = _dec(74)
        self.dec_v = _dec(35)

    def forward(self, z, x_t, x_a, x_v):
        # x_* are (B, 50, D) — pool to (B, D) as reconstruction target
        t_target = x_t.mean(dim=1)
        a_target = x_a.mean(dim=1)
        v_target = x_v.mean(dim=1)

        L_recon = (
            F.mse_loss(self.dec_t(z), t_target) +
            F.mse_loss(self.dec_a(z), a_target) +
            F.mse_loss(self.dec_v(z), v_target)
        ) / 3.0
        return L_recon


# ── Task Classifier ───────────────────────────────────────────
# Simple 2-layer MLP over z. Label smoothing handles
def get_class_weights(data_path, device):
    d = torch.load(data_path, map_location="cpu", weights_only=False)
    labels = np.array(d["labels"])
    counts = np.bincount(labels, minlength=cfg.num_classes).astype(float)
    # Sqrt smoothing — much gentler than raw inverse frequency
    weights = 1.0 / (np.sqrt(counts) + 1e-6)
    weights = weights / weights.mean()
    print("Class weights:", np.round(weights, 3))
    return torch.tensor(weights, dtype=torch.float32).to(device)


class TaskClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        dz = cfg.latent_dim
        self.net = nn.Sequential(
            nn.Linear(dz, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, cfg.num_classes),
        )

    def forward(self, z, labels=None):
        logits = self.net(z)
        if labels is not None:
            L_task = F.cross_entropy(logits, labels, label_smoothing=0.05)
            return logits, L_task
        return logits, None

In [13]:
# ============================================================
# Diffusion Prior (Noise Schedule + 1D U-Net + DDIM)
# ============================================================

# ── Cosine Noise Schedule ────────────────────────────────────

class DiffusionSchedule(nn.Module):
    def __init__(self, T=cfg.T):
        super().__init__()
        self.T = T
        t = torch.arange(T + 1, dtype=torch.float32)
        alpha_bar = torch.cos(((t / T) + 0.008) / 1.008 * torch.pi / 2) ** 2
        alpha_bar = alpha_bar / alpha_bar[0]
        betas     = (1 - alpha_bar[1:] / alpha_bar[:-1]).clamp(0, 0.999)
        alphas    = 1 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)

        self.register_buffer("betas",     betas)
        self.register_buffer("alphas",    alphas)
        self.register_buffer("alpha_bar", alpha_bar)
        self.register_buffer("sqrt_ab",   alpha_bar.sqrt())
        self.register_buffer("sqrt_1mab", (1 - alpha_bar).sqrt())

    def q_sample(self, z0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(z0)
        s_ab   = self.sqrt_ab[t].view(-1, 1)
        s_1mab = self.sqrt_1mab[t].view(-1, 1)
        return s_ab * z0 + s_1mab * noise, noise


# ── Residual Block for 1D U-Net ──────────────────────────────

class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, cond_dim):
        super().__init__()
        self.conv1  = nn.Conv1d(in_ch,  out_ch, 3, padding=1)
        self.conv2  = nn.Conv1d(out_ch, out_ch, 3, padding=1)
        self.norm1  = nn.GroupNorm(8, out_ch)
        self.norm2  = nn.GroupNorm(8, out_ch)
        self.act    = nn.SiLU()
        self.cond   = nn.Linear(cond_dim, out_ch)
        # skip projection if channels change
        self.skip   = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, c):
        # x : (B, in_ch, L)   c : (B, cond_dim)
        h = self.act(self.norm1(self.conv1(x)))
        h = h + self.cond(c).unsqueeze(-1)   # inject conditioning
        h = self.act(self.norm2(self.conv2(h)))
        return h + self.skip(x)


# ── 1D U-Net ─────────────────────────────────────────────────
# Operates on z reshaped to (B, 1, latent_dim) — treat latent
# dim as the sequence axis, 1 channel. Three encoder stages
# with channel multipliers 1x/2x/4x, symmetric decoder.
# Triple conditioning: timestep + class label + causal weights.

class UNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        dz    = cfg.latent_dim    # 128
        base  = 128
        cond  = 512               # conditioning dim

        # ── Conditioning projections ──────────────────────────
        # Sinusoidal timestep → MLP
        self.time_mlp = nn.Sequential(
            SinusoidalEmbed(base),
            nn.Linear(base, cond),
            nn.SiLU(),
            nn.Linear(cond, cond),
        )
        # Class label embedding (num_classes + 1 null token for CFG)
        self.class_emb = nn.Embedding(cfg.num_classes + 1, cond)
        # Causal weights w : (B, 3) → cond
        self.causal_mlp = nn.Sequential(
            nn.Linear(3, cond),
            nn.SiLU(),
            nn.Linear(cond, cond),
        )

        # ── Encoder ───────────────────────────────────────────
        self.enc1 = ResBlock1D(1,        base,    cond)   # (B, 128, 128)
        self.down1= nn.Conv1d(base,      base,    4, stride=2, padding=1)   # → 64

        self.enc2 = ResBlock1D(base,     base*2,  cond)   # (B, 256, 64)
        self.down2= nn.Conv1d(base*2,    base*2,  4, stride=2, padding=1)   # → 32

        self.enc3 = ResBlock1D(base*2,   base*4,  cond)   # (B, 512, 32)
        self.down3= nn.Conv1d(base*4,    base*4,  4, stride=2, padding=1)   # → 16

        # ── Bottleneck ────────────────────────────────────────
        self.mid1 = ResBlock1D(base*4,   base*4,  cond)
        self.mid2 = ResBlock1D(base*4,   base*4,  cond)

        # ── Decoder ───────────────────────────────────────────
        self.up3  = nn.ConvTranspose1d(base*4, base*4, 4, stride=2, padding=1)
        self.dec3 = ResBlock1D(base*4*2, base*2, cond)   # *2 for skip

        self.up2  = nn.ConvTranspose1d(base*2, base*2, 4, stride=2, padding=1)
        self.dec2 = ResBlock1D(base*2*2, base,  cond)

        self.up1  = nn.ConvTranspose1d(base,   base,   4, stride=2, padding=1)
        self.dec1 = ResBlock1D(base*2,   base,  cond)

        # ── Output head ───────────────────────────────────────
        self.out  = nn.Sequential(
            nn.GroupNorm(8, base),
            nn.SiLU(),
            nn.Conv1d(base, 1, 1),
        )

    def _cond(self, t, y, w):
        # Combine all three conditioning signals
        c = self.time_mlp(t) + self.class_emb(y) + self.causal_mlp(w)
        return c   # (B, cond)

    def forward(self, z, t, y, w):
        # z : (B, dz)   t : (B,) int   y : (B,) int   w : (B, 3)
        c = self._cond(t, y, w)
        x = z.unsqueeze(1)       # (B, 1, 128)

        # Encode
        e1 = self.enc1(x,  c)                   # (B, 128, 128)
        e2 = self.enc2(self.down1(e1), c)        # (B, 256, 64)
        e3 = self.enc3(self.down2(e2), c)        # (B, 512, 32)

        # Bottleneck
        h  = self.mid1(self.down3(e3), c)        # (B, 512, 16)
        h  = self.mid2(h, c)

        # Decode with skip connections
        h  = self.dec3(torch.cat([self.up3(h), e3], dim=1), c)
        h  = self.dec2(torch.cat([self.up2(h), e2], dim=1), c)
        h  = self.dec1(torch.cat([self.up1(h), e1], dim=1), c)

        return self.out(h).squeeze(1)            # (B, 128) predicted noise


# ── Sinusoidal Timestep Embedding ────────────────────────────

class SinusoidalEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t : (B,) int timesteps
        half = self.dim // 2
        freqs = torch.exp(
            -torch.arange(half, device=t.device).float()
            * (torch.log(torch.tensor(10000.0)) / (half - 1))
        )
        x = t.float().unsqueeze(1) * freqs.unsqueeze(0)   # (B, half)
        return torch.cat([x.sin(), x.cos()], dim=-1)       # (B, dim)


# ── Diffusion Training + DDIM Sampling ───────────────────────

class DiffusionPrior(nn.Module):
    def __init__(self):
        super().__init__()
        self.unet     = UNet1D()
        self.schedule = DiffusionSchedule()
        self.ema_unet = UNet1D()
        self.ema_unet.load_state_dict(self.unet.state_dict())
        self._ema_decay = 0.999

    @torch.no_grad()
    def update_ema(self):
        for p_ema, p in zip(self.ema_unet.parameters(), self.unet.parameters()):
            p_ema.data.mul_(self._ema_decay).add_(p.data, alpha=1 - self._ema_decay)

    def p_losses(self, z0, y, w):
        # z0 is stop-gradiented before entering here (done in AffectDiff.forward)
        B    = z0.shape[0]
        t    = torch.randint(0, cfg.T, (B,), device=z0.device)

        # Classifier-free guidance: randomly drop class label
        y_in = y.clone()
        drop = torch.rand(B, device=z0.device) < cfg.cfg_dropout
        y_in[drop] = cfg.num_classes   # null token

        zt, noise = self.schedule.q_sample(z0, t)
        pred      = self.unet(zt, t, y_in, w.detach())
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def ddim_sample(self, shape, y, w, steps=cfg.ddim_steps, scale=cfg.cfg_scale):
        # shape: (B, dz)
        device = next(self.ema_unet.parameters()).device
        z  = torch.randn(shape, device=device)
        ts = torch.linspace(cfg.T - 1, 0, steps, dtype=torch.long, device=device)

        for i, t_val in enumerate(ts):
            t_batch = t_val.expand(shape[0])
            ab      = self.schedule.alpha_bar[t_val]
            ab_prev = self.schedule.alpha_bar[ts[i + 1]] if i + 1 < len(ts) else torch.tensor(1.0)

            # CFG: conditional and unconditional predictions
            y_null  = torch.full_like(y, cfg.num_classes)
            eps_c   = self.ema_unet(z, t_batch, y,      w)
            eps_u   = self.ema_unet(z, t_batch, y_null, w)
            eps     = eps_u + scale * (eps_c - eps_u)

            # DDIM update (eta=0, deterministic)
            z0_pred = (z - (1 - ab).sqrt() * eps) / ab.sqrt()
            z       = ab_prev.sqrt() * z0_pred + (1 - ab_prev).sqrt() * eps

        return z

In [14]:
# ============================================================
# CELL 4 — AffectDiff (full model) + Loss Scheduler
# ============================================================

class AffectDiff(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.enc_t      = ModalityEncoder(300)
        self.enc_a      = ModalityEncoder(74)
        self.enc_v      = ModalityEncoder(35)
        self.causal     = CausalAttentionGraph()
        self.vae        = VAEBottleneck()
        self.decoder    = ReconDecoder()
        self.classifier = TaskClassifier()
        self.diffusion  = DiffusionPrior()

    def forward(self, x_t, x_a, x_v, labels=None):
        h_t = self.enc_t(x_t)
        h_a = self.enc_a(x_a)
        h_v = self.enc_v(x_v)

        A, w, L_causal = self.causal(h_t, h_a, h_v)
        z, mu, log_var, L_kl = self.vae(h_t, h_a, h_v, w)
        logits, L_task = self.classifier(z, labels)
        L_diff = torch.tensor(0.0, device=z.device)
        L_recon = self.decoder(z, x_t, x_a, x_v)

        losses = dict(
            task   = L_task.mean(),
            kl     = L_kl.mean(),
            diff   = L_diff.mean(),
            recon  = L_recon.mean(),
            causal = L_causal.mean(),
        )
        return logits, losses, w

# ── Loss Scheduler ───────────────────────────────────────────
# Curriculum warmup: classification dominates early training.
# KL ramps slower (30 epochs) to prevent posterior collapse.
# Auxiliary losses (diff, recon, causal) ramp over 20 epochs.

def compute_loss(losses, epoch):
    gamma    = min(1.0, epoch / cfg.warmup_aux)
    gamma_kl = min(1.0, epoch / cfg.warmup_kl)

    total = (
        losses["task"]
        + gamma_kl * cfg.lam_kl_beta * losses["kl"]
        + gamma    * cfg.lam_diff    * losses["diff"]
        + gamma    * cfg.lam_recon   * losses["recon"]
        +            cfg.lam_causal  * losses["causal"]
    )
    return total


# ============================================================
# CELL 5 — Training + Evaluation Loop
# ============================================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import f1_score, accuracy_score
import time

def train_one_epoch(model, loader, optimizer, scaler, epoch):
    model.train()
    total_loss = 0.0
    for x_t, x_a, x_v, labels in loader:
        x_t, x_a, x_v, labels = (
            x_t.to(cfg.device),
            x_a.to(cfg.device),
            x_v.to(cfg.device),
            labels.to(cfg.device),
        )
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            logits, losses, _ = model(x_t, x_a, x_v, labels)
            # Sum individual loss tensors — each is already a scalar mean
            # from its replica, DataParallel stacks them into (n_gpu,)
            # vectors which we mean here before computing total
            losses = {k: v.mean() for k, v in losses.items()}
            loss   = compute_loss(losses, epoch)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        m = model.module if hasattr(model, "module") else model
        m.diffusion.update_ema()
        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, verbose=False):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    for x_t, x_a, x_v, labels in loader:
        x_t, x_a, x_v, labels = (
            x_t.to(cfg.device),
            x_a.to(cfg.device),
            x_v.to(cfg.device),
            labels.to(cfg.device),
        )
        with torch.amp.autocast("cuda"):
            logits, losses, _ = model(x_t, x_a, x_v, labels)
            losses = {k: v.mean() for k, v in losses.items()}
            loss   = compute_loss(losses, epoch=100)

        total_loss += loss.item()
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    wf1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0)

    if verbose:
        classes = ["Happy", "Sad", "Angry"][:cfg.num_classes]
        per_class = f1_score(all_labels, all_preds, average=None, zero_division=0)
        for c, f in zip(classes, per_class):
            print(f"  {c}: F1={f:.3f}")

    return total_loss / len(loader), acc, wf1



# ── Fixed GradScaler in run_training ────────────────────────

def run_training():
    train_loader, val_loader, test_loader = get_loaders(cfg.data_path)
    # class_weights = get_class_weights(cfg.data_path, cfg.device)
    model = AffectDiff().to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_acc   = 0.0
    best_val_wf1   = 0.0
    patience_count = 0
    patience       = 25   # was 20, give it more room

    print(f"Training on {cfg.device} | {torch.cuda.device_count()} GPU(s)")
    print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}\n")

    for epoch in range(1, cfg.epochs + 1):
        t0 = time.time()
        train_loss             = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_acc, val_wf1 = evaluate(model, val_loader)
        scheduler.step()

        elapsed = time.time() - t0
        print(
            f"Epoch {epoch:03d} | "
            f"train {train_loss:.4f} | "
            f"val {val_loss:.4f} | "
            f"acc {val_acc:.4f} | "
            f"wF1 {val_wf1:.4f} | "
            f"{elapsed:.1f}s"
        )

        if val_wf1 > best_val_wf1:
            best_val_wf1   = val_wf1
            patience_count = 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, "/kaggle/working/affect_diff_best.pt")
            print(f"  --> saved (val_wF1 {val_wf1:.4f})")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    print("\nLoading best checkpoint for test evaluation...")
    best_model = AffectDiff().to(cfg.device)
    best_model.load_state_dict(torch.load("/kaggle/working/affect_diff_best.pt"))
    test_loss, test_acc, test_wf1 = evaluate(best_model, test_loader, verbose=True)
    print(f"\nTest | loss {test_loss:.4f} | acc {test_acc:.4f} | wF1 {test_wf1:.4f}")

    return best_model

# ── Kick off training ─────────────────────────────────────────
# Comment this out if you want to inspect the model first.
import warnings
warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0")
best_model = run_training()

  Happy: 1850 (70.9%)
  Sad: 482 (18.5%)
  Angry: 279 (10.7%)


/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


Training on cuda | 2 GPU(s)
Train: 1827 | Val: 391 | Test: 393

Epoch 001 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 3.5s
  --> saved (val_wF1 0.5139)


/tmp/ipykernel_55/2448028006.py:155: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 002 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.9s
Epoch 003 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 004 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.6s
Epoch 005 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 006 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 007 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 008 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 009 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.8s
Epoch 010 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 2.0s
Epoch 011 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 012 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 013 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 014 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 015 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 016 | train nan | val nan | acc 0.5575 | wF1 0.5139 | 1.7s
Epoch 017 | train nan | v

/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


  Happy: F1=0.770
  Sad: F1=0.048
  Angry: F1=0.101

Test | loss nan | acc 0.6260 | wF1 0.5797


In [15]:
@torch.no_grad()
def diagnose(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for x_t, x_a, x_v, labels in loader:
        x_t, x_a, x_v, labels = (
            x_t.to(cfg.device),
            x_a.to(cfg.device),
            x_v.to(cfg.device),
            labels.to(cfg.device),
        )
        with torch.amp.autocast("cuda"):
            logits, _, _ = model(x_t, x_a, x_v, labels)
        all_preds.extend(logits.argmax(dim=-1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    classes = ["Happy", "Sad", "Angry", "Other"]
    print("\nPrediction distribution:")
    pred_counts = np.bincount(all_preds, minlength=4)
    true_counts = np.bincount(all_labels, minlength=4)
    for c, p, t in zip(classes, pred_counts, true_counts):
        print(f"  {c}: predicted={p}, actual={t}")

best_model = AffectDiff().to(cfg.device)
best_model.load_state_dict(torch.load("/kaggle/working/affect_diff_best.pt"))
_, val_loader, test_loader = get_loaders(cfg.data_path)
diagnose(best_model, test_loader)

/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


  Happy: 1850 (70.9%)
  Sad: 482 (18.5%)
  Angry: 279 (10.7%)

Prediction distribution:
  Happy: predicted=337, actual=286
  Sad: predicted=21, actual=63
  Angry: predicted=35, actual=44
  Other: predicted=0, actual=0


In [16]:
# ── Ablation runner ───────────────────────────────────────────
# Run this cell three times, changing ablation_mode each time

ablation_mode = "full"   # options: "full", "no_diffusion", "no_causal"

# Patch AffectDiff.forward based on mode
class AffectDiffAblation(nn.Module):
    def __init__(self, mode="full"):
        super().__init__()
        self.mode = mode
        self.enc_t      = ModalityEncoder(300)
        self.enc_a      = ModalityEncoder(74)
        self.enc_v      = ModalityEncoder(35)
        self.causal     = CausalAttentionGraph()
        self.vae        = VAEBottleneck()
        self.decoder    = ReconDecoder()
        self.classifier = TaskClassifier()
        self.diffusion  = DiffusionPrior()

    def forward(self, x_t, x_a, x_v, labels=None):
        h_t = self.enc_t(x_t)
        h_a = self.enc_a(x_a)
        h_v = self.enc_v(x_v)

        if self.mode == "no_causal":
            # Uniform weights — no causal graph
            w = torch.ones(x_t.shape[0], 3, device=x_t.device) / 3.0
            L_causal = torch.tensor(0.0, device=x_t.device)
        else:
            A, w, L_causal = self.causal(h_t, h_a, h_v)

        z, mu, log_var, L_kl = self.vae(h_t, h_a, h_v, w)
        logits, L_task = self.classifier(z, labels)

        if self.mode == "no_diffusion" or self.mode == "no_causal":
            L_diff = torch.tensor(0.0, device=z.device)
        else:
            L_diff = self.diffusion.p_losses(z.detach(), labels, w) if labels is not None else torch.tensor(0.0, device=z.device)

        L_recon = self.decoder(z, x_t, x_a, x_v)

        losses = dict(
            task   = L_task.mean(),
            kl     = L_kl.mean(),
            diff   = L_diff.mean() if hasattr(L_diff, 'mean') else L_diff,
            recon  = L_recon.mean(),
            causal = L_causal.mean() if hasattr(L_causal, 'mean') else L_causal,
        )
        return logits, losses, w


def run_ablation(mode):
    print(f"\n{'='*50}")
    print(f"Ablation: {mode}")
    print(f"{'='*50}")

    train_loader, val_loader, test_loader = get_loaders(cfg.data_path)
    model = AffectDiffAblation(mode=mode).to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_wf1   = 0.0
    patience_count = 0
    patience       = 25

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_acc, val_wf1 = evaluate(model, val_loader)
        scheduler.step()

        print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f} | wF1 {val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1   = val_wf1
            patience_count = 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, f"/kaggle/working/ablation_{mode}.pt")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    best = AffectDiffAblation(mode=mode).to(cfg.device)
    best.load_state_dict(torch.load(f"/kaggle/working/ablation_{mode}.pt"))
    test_loss, test_acc, test_wf1 = evaluate(best, test_loader, verbose=True)
    print(f"\n[{mode}] Test | acc {test_acc:.4f} | wF1 {test_wf1:.4f}")
    return test_acc, test_wf1


# Run all three
results = {}
for mode in ["no_causal", "no_diffusion", "full"]:
    acc, wf1 = run_ablation(mode)
    results[mode] = {"acc": acc, "wf1": wf1}

print("\n\nAblation Summary")
print(f"{'Mode':<20} {'Acc':>8} {'wF1':>8}")
print("-" * 38)
for mode, r in results.items():
    print(f"{mode:<20} {r['acc']:>8.4f} {r['wf1']:>8.4f}")


Ablation: no_causal
  Happy: 1850 (70.9%)
  Sad: 482 (18.5%)
  Angry: 279 (10.7%)


/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


Epoch 001 | train nan | val nan | acc 0.4859 | wF1 0.4833


/tmp/ipykernel_55/2973284998.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 002 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 003 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 004 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 005 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 006 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 007 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 008 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 009 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 010 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 011 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 012 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 013 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 014 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 015 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 016 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 017 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 018 | train nan | val nan | acc 0.4859 | wF1 0.4833
Epoch 019 | tr

...

Epoch 001 | train nan | val nan | acc 0.5627 | wF1 0.5245


/tmp/ipykernel_55/2973284998.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 002 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 003 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 004 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 005 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 006 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 007 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 008 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 009 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 010 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 011 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 012 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 013 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 014 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 015 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 016 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 017 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 018 | train nan | val nan | acc 0.5627 | wF1 0.5245
Epoch 019 | tr

/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


  Happy: F1=0.770
  Sad: F1=0.062
  Angry: F1=0.100

[full] Test | acc 0.6158 | wF1 0.5819


Ablation Summary
Mode                      Acc      wF1
--------------------------------------
no_causal              0.5165   0.5405
no_diffusion           0.4656   0.5029
full                   0.6158   0.5819


In [17]:
# ── Ablation runner ───────────────────────────────────────────
# Run this cell three times, changing ablation_mode each time

ablation_mode = "full"   # options: "full", "no_diffusion", "no_causal"

# Patch AffectDiff.forward based on mode
class AffectDiffAblation(nn.Module):
    def __init__(self, mode="full"):
        super().__init__()
        self.mode = mode
        self.enc_t      = ModalityEncoder(300)
        self.enc_a      = ModalityEncoder(74)
        self.enc_v      = ModalityEncoder(35)
        self.causal     = CausalAttentionGraph()
        self.vae        = VAEBottleneck()
        self.decoder    = ReconDecoder()
        self.classifier = TaskClassifier()
        self.diffusion  = DiffusionPrior()

    def forward(self, x_t, x_a, x_v, labels=None):
        h_t = self.enc_t(x_t)
        h_a = self.enc_a(x_a)
        h_v = self.enc_v(x_v)

        if self.mode == "no_causal":
            # Uniform weights — no causal graph
            w = torch.ones(x_t.shape[0], 3, device=x_t.device) / 3.0
            L_causal = torch.tensor(0.0, device=x_t.device)
        else:
            A, w, L_causal = self.causal(h_t, h_a, h_v)

        z, mu, log_var, L_kl = self.vae(h_t, h_a, h_v, w)
        logits, L_task = self.classifier(z, labels)

        if self.mode == "no_diffusion" or self.mode == "no_causal":
            L_diff = torch.tensor(0.0, device=z.device)
        else:
            L_diff = self.diffusion.p_losses(z.detach(), labels, w) if labels is not None else torch.tensor(0.0, device=z.device)

        L_recon = self.decoder(z, x_t, x_a, x_v)

        losses = dict(
            task   = L_task.mean(),
            kl     = L_kl.mean(),
            diff   = L_diff.mean() if hasattr(L_diff, 'mean') else L_diff,
            recon  = L_recon.mean(),
            causal = L_causal.mean() if hasattr(L_causal, 'mean') else L_causal,
        )
        return logits, losses, w


def run_ablation(mode):
    print(f"\n{'='*50}")
    print(f"Ablation: {mode}")
    print(f"{'='*50}")

    train_loader, val_loader, test_loader = get_loaders(cfg.data_path)
    model = AffectDiffAblation(mode=mode).to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_wf1   = 0.0
    patience_count = 0
    patience       = 25

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_acc, val_wf1 = evaluate(model, val_loader)
        scheduler.step()

        print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f} | wF1 {val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1   = val_wf1
            patience_count = 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, f"/kaggle/working/ablation_{mode}.pt")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    best = AffectDiffAblation(mode=mode).to(cfg.device)
    best.load_state_dict(torch.load(f"/kaggle/working/ablation_{mode}.pt"))
    test_loss, test_acc, test_wf1 = evaluate(best, test_loader, verbose=True)
    print(f"\n[{mode}] Test | acc {test_acc:.4f} | wF1 {test_wf1:.4f}")
    return test_acc, test_wf1


# Run all three
results = {}
for mode in ["no_causal", "no_diffusion", "full"]:
    acc, wf1 = run_ablation(mode)
    results[mode] = {"acc": acc, "wf1": wf1}

print("\n\nAblation Summary")
print(f"{'Mode':<20} {'Acc':>8} {'wF1':>8}")
print("-" * 38)
for mode, r in results.items():
    print(f"{mode:<20} {r['acc']:>8.4f} {r['wf1']:>8.4f}")


Ablation: no_causal
  Happy: 1850 (70.9%)
  Sad: 482 (18.5%)
  Angry: 279 (10.7%)


/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2
/tmp/ipykernel_55/2973284998.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 001 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 002 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 003 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 004 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 005 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 006 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 007 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 008 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 009 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 010 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 011 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 012 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 013 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 014 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 015 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 016 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 017 | train nan | val nan | acc 0.4731 | wF1 0.4771
Epoch 018 | tr

/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


  Happy: F1=0.691
  Sad: F1=0.212
  Angry: F1=0.091

[no_causal] Test | acc 0.5318 | wF1 0.5473

Ablation: no_diffusion
  Happy: 1850 (70.9%)
  Sad: 482 (18.5%)
  Angry: 279 (10.7%)


/tmp/ipykernel_55/4099703319.py:157: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # was 2


Epoch 001 | train nan | val nan | acc 0.4604 | wF1 0.4597


/tmp/ipykernel_55/2973284998.py:73: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


KeyboardInterrupt: 

In [ ]:
# ── Ablation runner ───────────────────────────────────────────
# Run this cell three times, changing ablation_mode each time

ablation_mode = "no_diffusion"   # options: "full", "no_diffusion", "no_causal"

# Patch AffectDiff.forward based on mode
class AffectDiffAblation(nn.Module):
    def __init__(self, mode="full"):
        super().__init__()
        self.mode = mode
        self.enc_t      = ModalityEncoder(300)
        self.enc_a      = ModalityEncoder(74)
        self.enc_v      = ModalityEncoder(35)
        self.causal     = CausalAttentionGraph()
        self.vae        = VAEBottleneck()
        self.decoder    = ReconDecoder()
        self.classifier = TaskClassifier()
        self.diffusion  = DiffusionPrior()

    def forward(self, x_t, x_a, x_v, labels=None):
        h_t = self.enc_t(x_t)
        h_a = self.enc_a(x_a)
        h_v = self.enc_v(x_v)

        if self.mode == "no_causal":
            # Uniform weights — no causal graph
            w = torch.ones(x_t.shape[0], 3, device=x_t.device) / 3.0
            L_causal = torch.tensor(0.0, device=x_t.device)
        else:
            A, w, L_causal = self.causal(h_t, h_a, h_v)

        z, mu, log_var, L_kl = self.vae(h_t, h_a, h_v, w)
        logits, L_task = self.classifier(z, labels)

        if self.mode == "no_diffusion" or self.mode == "no_causal":
            L_diff = torch.tensor(0.0, device=z.device)
        else:
            L_diff = self.diffusion.p_losses(z.detach(), labels, w) if labels is not None else torch.tensor(0.0, device=z.device)

        L_recon = self.decoder(z, x_t, x_a, x_v)

        losses = dict(
            task   = L_task.mean(),
            kl     = L_kl.mean(),
            diff   = L_diff.mean() if hasattr(L_diff, 'mean') else L_diff,
            recon  = L_recon.mean(),
            causal = L_causal.mean() if hasattr(L_causal, 'mean') else L_causal,
        )
        return logits, losses, w


def run_ablation(mode):
    print(f"\n{'='*50}")
    print(f"Ablation: {mode}")
    print(f"{'='*50}")

    train_loader, val_loader, test_loader = get_loaders(cfg.data_path)
    model = AffectDiffAblation(mode=mode).to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_wf1   = 0.0
    patience_count = 0
    patience       = 25

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_acc, val_wf1 = evaluate(model, val_loader)
        scheduler.step()

        print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f} | wF1 {val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1   = val_wf1
            patience_count = 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, f"/kaggle/working/ablation_{mode}.pt")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    best = AffectDiffAblation(mode=mode).to(cfg.device)
    best.load_state_dict(torch.load(f"/kaggle/working/ablation_{mode}.pt"))
    test_loss, test_acc, test_wf1 = evaluate(best, test_loader, verbose=True)
    print(f"\n[{mode}] Test | acc {test_acc:.4f} | wF1 {test_wf1:.4f}")
    return test_acc, test_wf1


# Run all three
results = {}
for mode in ["no_causal", "no_diffusion", "full"]:
    acc, wf1 = run_ablation(mode)
    results[mode] = {"acc": acc, "wf1": wf1}

print("\n\nAblation Summary")
print(f"{'Mode':<20} {'Acc':>8} {'wF1':>8}")
print("-" * 38)
for mode, r in results.items():
    print(f"{mode:<20} {r['acc']:>8.4f} {r['wf1']:>8.4f}")

In [ ]:
# ── Ablation runner ───────────────────────────────────────────
# Run this cell three times, changing ablation_mode each time

ablation_mode = "no_causal"   # options: "full", "no_diffusion", "no_causal"

# Patch AffectDiff.forward based on mode
class AffectDiffAblation(nn.Module):
    def __init__(self, mode="full"):
        super().__init__()
        self.mode = mode
        self.enc_t      = ModalityEncoder(300)
        self.enc_a      = ModalityEncoder(74)
        self.enc_v      = ModalityEncoder(35)
        self.causal     = CausalAttentionGraph()
        self.vae        = VAEBottleneck()
        self.decoder    = ReconDecoder()
        self.classifier = TaskClassifier()
        self.diffusion  = DiffusionPrior()

    def forward(self, x_t, x_a, x_v, labels=None):
        h_t = self.enc_t(x_t)
        h_a = self.enc_a(x_a)
        h_v = self.enc_v(x_v)

        if self.mode == "no_causal":
            # Uniform weights — no causal graph
            w = torch.ones(x_t.shape[0], 3, device=x_t.device) / 3.0
            L_causal = torch.tensor(0.0, device=x_t.device)
        else:
            A, w, L_causal = self.causal(h_t, h_a, h_v)

        z, mu, log_var, L_kl = self.vae(h_t, h_a, h_v, w)
        logits, L_task = self.classifier(z, labels)

        if self.mode == "no_diffusion" or self.mode == "no_causal":
            L_diff = torch.tensor(0.0, device=z.device)
        else:
            L_diff = self.diffusion.p_losses(z.detach(), labels, w) if labels is not None else torch.tensor(0.0, device=z.device)

        L_recon = self.decoder(z, x_t, x_a, x_v)

        losses = dict(
            task   = L_task.mean(),
            kl     = L_kl.mean(),
            diff   = L_diff.mean() if hasattr(L_diff, 'mean') else L_diff,
            recon  = L_recon.mean(),
            causal = L_causal.mean() if hasattr(L_causal, 'mean') else L_causal,
        )
        return logits, losses, w


def run_ablation(mode):
    print(f"\n{'='*50}")
    print(f"Ablation: {mode}")
    print(f"{'='*50}")

    train_loader, val_loader, test_loader = get_loaders(cfg.data_path)
    model = AffectDiffAblation(mode=mode).to(cfg.device)
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)
    scaler    = torch.amp.GradScaler("cuda")

    best_val_wf1   = 0.0
    patience_count = 0
    patience       = 25

    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler, epoch)
        val_loss, val_acc, val_wf1 = evaluate(model, val_loader)
        scheduler.step()

        print(f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | acc {val_acc:.4f} | wF1 {val_wf1:.4f}")

        if val_wf1 > best_val_wf1:
            best_val_wf1   = val_wf1
            patience_count = 0
            state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
            torch.save(state, f"/kaggle/working/ablation_{mode}.pt")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

    best = AffectDiffAblation(mode=mode).to(cfg.device)
    best.load_state_dict(torch.load(f"/kaggle/working/ablation_{mode}.pt"))
    test_loss, test_acc, test_wf1 = evaluate(best, test_loader, verbose=True)
    print(f"\n[{mode}] Test | acc {test_acc:.4f} | wF1 {test_wf1:.4f}")
    return test_acc, test_wf1


# Run all three
results = {}
for mode in ["no_causal", "no_diffusion", "full"]:
    acc, wf1 = run_ablation(mode)
    results[mode] = {"acc": acc, "wf1": wf1}

print("\n\nAblation Summary")
print(f"{'Mode':<20} {'Acc':>8} {'wF1':>8}")
print("-" * 38)
for mode, r in results.items():
    print(f"{mode:<20} {r['acc']:>8.4f} {r['wf1']:>8.4f}")